In [0]:
df = spark.read.table('data.orders.occupazione').limit(3)
display(df)

In [0]:
# Find average employment (occupazione) per country. 
import pyspark.sql.functions as f 
df = spark.read.table('data.orders.occupazione')
df = df.groupBy('country')\
    .agg(f.round(f.avg('obs_value'),2).alias('avg_emp_rate'))\
    .orderBy("country")

display(df)

In [0]:
# Find max unemployment per year. 
import pyspark.sql.functions as f
df = spark.read.table('data.orders.disoccupazione')
df = df.groupBy('year')\
    .agg(f.max('obs_value').alias('max_unemp_rate'))\
    .orderBy("year")

display(df)
#

In [0]:
# Group by sex and calculate average obs_value.
df = spark.read.table('data.orders.disoccupazione')
df = df.groupBy('sex')\
    .agg(f.round(f.avg('obs_value'),2).alias('avg_unemp_rate'))\
    .orderBy("sex")
display(df)


In [0]:
# Find countries where unemployment > 10. 
df = spark.read.table('data.orders.disoccupazione')
df = df.filter(df['obs_value'] > 10)\
    .select('country')\
    .distinct()
display(df)

In [0]:
# Join both tables on: iso_code, country, sex, age, year
import pyspark.sql.functions as f
d = spark.read.table('data.orders.disoccupazione').alias('d')
o = spark.read.table('data.orders.occupazione').alias('o')
df = d.join(o, 
    (f.col('d.iso_code') == f.col('o.iso_code')) &
    (f.col('d.country') == f.col('o.country')) &
    (f.col('d.sex') == f.col('o.sex')) &
    (f.col('d.age') == f.col('o.age')) &
    (f.col('d.year') == f.col('o.year')) 
    , 'inner')\
    .select('d.iso_code', 'd.country', 'd.sex', 
        'd.age', 'd.year', f.col('d.obs_value').alias('disoccupazione'), f.col('o.obs_value').alias('occupazione'))
display(df)

In [0]:
# Join both tables on: iso_code, country, sex, age, year
import pyspark.sql.functions as f
d = spark.read.table('data.orders.disoccupazione').alias('d')
o = spark.read.table('data.orders.occupazione').alias('o')
df = d.join(o, 
    (f.col('d.iso_code') == f.col('o.iso_code')) &
    (f.col('d.country') == f.col('o.country')) &
    (f.col('d.sex') == f.col('o.sex')) &
    (f.col('d.age') == f.col('o.age')) &
    (f.col('d.year') == f.col('o.year')) 
    , 'inner')\
    .select('d.iso_code', 'd.country', 'd.sex', 
        'd.age', 'd.year', f.col('d.obs_value').alias('disoccupazione'), f.col('o.obs_value').alias('occupazione'))
display(df)

In [0]:
import pyspark.sql.functions as f
d = spark.read.table('data.orders.disoccupazione').alias('d')
o = spark.read.table('data.orders.occupazione').alias('o')
df = d.join(o, 
    (f.col('d.iso_code') == f.col('o.iso_code')) &
    (f.col('d.country') == f.col('o.country')) &
    (f.col('d.sex') == f.col('o.sex')) &
    (f.col('d.age') == f.col('o.age')) &
    (f.col('d.year') == f.col('o.year')) 
    , 'inner')\
    .where(f.col('d.obs_value') > f.col('o.obs_value'))\
    .select( 'd.country', 'd.sex', 
        'd.age', 'd.year', f.col('d.obs_value').alias('unemployment'), f.col('o.obs_value').alias('employment'))
display(df)

In [0]:
# Compute difference: employment - unemployment
import pyspark.sql.functions as f
d = spark.read.table('data.orders.disoccupazione').alias('d')
o = spark.read.table('data.orders.occupazione').alias('o')
df = d.join(o, 
            (f.col('d.iso_code') == f.col('o.iso_code')) &
            (f.col('d.country') == f.col('o.country')) &
            (f.col('d.sex') == f.col('o.sex')) &
            (f.col('d.age') == f.col('o.age')) &
            (f.col('d.year') == f.col('o.year')),
            'inner')\
    .withColumn('gap', f.round(f.col('o.obs_value') - f.col('d.obs_value'),2))\
    .select( 'd.country', 'd.sex','d.age', 'd.year', 
            f.col('d.obs_value').alias('unemployment'), f.col('o.obs_value').alias('employment')
            ,'gap')\
    .orderBy('year', 'country')
display(df)

In [0]:
# Find top 5 countries with highest unemployment gap.
import pyspark.sql.functions as f
d = spark.read.table('data.orders.disoccupazione').alias('d')
o = spark.read.table('data.orders.occupazione').alias('o')
df = d.join(o, 
            (f.col('d.iso_code') == f.col('o.iso_code')) &
            (f.col('d.country') == f.col('o.country')) &
            (f.col('d.sex') == f.col('o.sex')) &
            (f.col('d.age') == f.col('o.age')) &
            (f.col('d.year') == f.col('o.year')),
            'inner')\
    .groupBy('d.country','d.year')\
    .agg(f.sum(f.col('o.obs_value')).alias('employment'), f.sum(f.col('d.obs_value')).alias('unemployment'))\
    .withColumn('gap', f.round(f.col('employment') - f.col('unemployment'),2))

df = df.where(f.col('gap') < 0)\
    .orderBy('gap')\
    .limit(5)
display(df)

In [0]:
# Use window function to rank countries by unemployment per year. 
import pyspark.sql.functions as f
from pyspark.sql.window import Window
d = spark.read.table('data.orders.disoccupazione')
w = Window.partitionBy('year').orderBy(f.desc('unemployment'))

df = d.groupBy('year','country').agg(f.round(f.sum(f.col('obs_value')),2).alias('unemployment'))
df = df.withColumn('rank', f.rank().over(w))
display(df)


In [0]:
# Calculate employment ratio:
import pyspark.sql.functions as f
d = spark.read.table('data.orders.disoccupazione').alias('d')
o = spark.read.table('data.orders.occupazione').alias('o')
df = d.join(o, 
            (f.col('d.iso_code') == f.col('o.iso_code')) &
            (f.col('d.country') == f.col('o.country')) &
            (f.col('d.sex') == f.col('o.sex')) &
            (f.col('d.age') == f.col('o.age')) &
            (f.col('d.year') == f.col('o.year')),
            'inner')\
    .groupBy('d.country','d.year')\
    .agg(f.sum(f.col('o.obs_value')).alias('employment'), f.sum(f.col('d.obs_value')).alias('unemployment'))\
    .withColumn('parti', f.round(f.col('employment') + f.col('unemployment'),2))

df = df.withColumn('ratio', f.round(f.col('unemployment') / f.col('parti'),2))\
    .orderBy("country", "year")
display(df)

In [0]:
# Find unemployment percentage per country 
import pyspark.sql.functions as f
d = spark.read.table('data.orders.disoccupazione').alias('d')
o = spark.read.table('data.orders.occupazione').alias('o')
df = d.join(o, 
            (f.col('d.iso_code') == f.col('o.iso_code')) &
            (f.col('d.country') == f.col('o.country')) &
            (f.col('d.sex') == f.col('o.sex')) &
            (f.col('d.age') == f.col('o.age')) &
            (f.col('d.year') == f.col('o.year')),
            'inner')\
    .groupBy('d.country','d.year')\
    .agg(f.sum(f.col('o.obs_value')).alias('employment'), f.sum(f.col('d.obs_value')).alias('unemployment'))\
    .withColumn('parti', f.round(f.col('employment') + f.col('unemployment'),2))

df = df.withColumn('unemp_per', f.round(f.col('unemployment') *100 / f.col('parti'),2))\
    .orderBy("country", "year")
display(df)

In [0]:
'''•	Label countries as: 
	"High Unemployment" if obs_value > threshold 
	"Low Unemployment" otherwise 
'''
import pyspark.sql.functions as f
d = spark.read.table('data.orders.disoccupazione')
t = d.agg(f.avg('obs_value')).collect()[0][0]

df2 = d.withColumn('label', f.when(f.col('obs_value') > t, 'High Unemployment')
    .otherwise('Low Unemployment'))\
    .orderBy("country", "year")
display(df2)

In [0]:
# Calculate year-over-year change in obs_value. 
import pyspark.sql.functions as f
from pyspark.sql.window import Window
d = spark.read.table('data.orders.disoccupazione').alias('d')
w = Window.partitionBy('country').orderBy('year')

d = spark.read.table('data.orders.disoccupazione')
t = d.groupBy("country", "year").agg(f.round(f.sum('obs_value'),2).alias('unemployment'))

df = t.withColumn('prev', f.lag('unemployment').over(w))\
    .withColumn('change', f.round(f.col('unemployment') - f.col('prev'),2))
display(df)

In [0]:
# Find growth rate of employment year over year 
import pyspark.sql.functions as f
from pyspark.sql.window import Window
d = spark.read.table('data.orders.occupazione')
w = Window.partitionBy('country').orderBy('year')

t = d.groupBy("country", "year").agg(f.round(f.sum('obs_value'),2).alias('employment'))

df = t.withColumn('prev', f.lag('employment').over(w))\
    .withColumn('change', f.round((f.col('employment') - f.col('prev'))*100 / f.when(f.col('prev') != 0, f.col('prev')).otherwise(None), 2))
display(df)

In [0]:
# Identify years where unemployment increased compared to previous year.  
import pyspark.sql.functions as f
from pyspark.sql.window import Window
d = spark.read.table('data.orders.disoccupazione').alias('d')
w = Window.partitionBy('country').orderBy('year')

d = spark.read.table('data.orders.disoccupazione')
t = d.groupBy("country", "year").agg(f.round(f.sum('obs_value'),2).alias('unemployment'))

df = t.withColumn('prev', f.lag('unemployment').over(w))\
    .where(f.col('unemployment') > f.col('prev'))
display(df)

In [0]:
# Pivot sex column → male vs female comparison
import pyspark.sql.functions as f
d = spark.read.table('data.orders.disoccupazione').alias('d')
o = spark.read.table('data.orders.occupazione').alias('o')
df = d.join(o, 
            (f.col('d.iso_code') == f.col('o.iso_code')) &
            (f.col('d.country') == f.col('o.country')) &
            (f.col('d.sex') == f.col('o.sex')) &
            (f.col('d.age') == f.col('o.age')) &
            (f.col('d.year') == f.col('o.year')),
            'inner')\
    .select( 'd.country', 'd.sex','d.age', 'd.year', 
            f.col('d.obs_value').alias('unemployment'), f.col('o.obs_value').alias('employment'))

df2 = df.withColumn('male_emp', f.when(f.col('sex') == 'Male', f.col('employment')).otherwise(0))\
    .withColumn('female_emp', f.when(f.col('sex') == 'Female', f.col('employment')).otherwise(0))\
    .withColumn('male_unemp', f.when(f.col('sex') == 'Male', f.col('unemployment')).otherwise(0))\
    .withColumn('female_unemp', f.when(f.col('sex') == 'Female', f.col('unemployment')).otherwise(0))\
    .groupBy('country', 'year')\
    .agg(f.round(f.sum(f.col('male_emp')),2).alias('male_emp'), 
         f.round(f.sum(f.col('female_emp')),2).alias('female_emp'),
         f.round(f.sum(f.col('male_unemp')),2).alias('male_unemp'), 
         f.round(f.sum(f.col('female_unemp')),2).alias('female_unemp'))\
    .select("country", "year", "male_emp", "female_emp", "male_unemp", "female_unemp")\
    .orderBy("country", "year")
display(df2)

df3 = df.groupBy("country", "year")\
    .pivot("sex", ["Male", "Female"])\
    .agg(
        f.round(f.sum("employment"), 2).alias("emp"),
        f.round(f.sum("unemployment"), 2).alias("unemp")
    )\
    .orderBy("country", "year")
display(df3)